In [1]:
from coniferest.pineforest import PineForest
from coniferest.aadforest import AADForest
from coniferest.isoforest import IsolationForest
from coniferest.session import Session
from coniferest.session.callback import TerminateAfter, viewer_decision_callback
import numpy as np
import time
import pandas as pd
import matplotlib.pyplot as plt
from coniferest.aadforest import AADForest
from coniferest.label import Label

import sys
sys.path.insert(0, '../..')

from sn_clf.scripts.utils import load_features, plot_config

plot_config()

In [2]:
#oids, features = load_features('../../dr23-features/sid_snad_clf_r_100.dat', '../../dr23-features/feature_sn_hard.dat')
field = 824

oids, features = load_features(f'../../dr23-features/collected_by_field/dr23_oid_{field}.dat', 
                                f'../../dr23-features/collected_by_field_sn_hard/dr23_feature_{field}.dat')

# _, rb_features = load_features(f'../../dr23-features/collected_by_field/dr23_oid_{field}.dat', 
#                                f'../../dr23-features/collected_by_field_rb/dr23_feature_{field}.dat')

#_, rb_features = load_features(f'../../dr23-features/sid_snad_clf_r_100.dat', 
#                               f'../../dr23-features/feature_rb.dat')


#features = np.hstack([features[:,-1].reshape(-1,1), rb_features[:,-1].reshape(-1,1)])
print(features.shape)

rs = 1
train_data = pd.read_csv('../data/train_data_big.csv')

feature_names = '../../dr23-features/feature_sn_hard.name'
with open(feature_names) as f:
    names = f.read().split()

(557903, 48)


In [3]:
artefact_oid = np.array(train_data[train_data['artefact'] == 1]['oid'], dtype=np.uint64)
mask = ~np.isin(artefact_oid, oids)
artefact_oid = artefact_oid[mask][:10]
art_priors = np.array(train_data[train_data['artefact'] == 1][names], dtype=np.float32)[mask][:10]

In [3]:
sn_oid = np.array(train_data[train_data['is_sn'] == 1]['oid'], dtype=np.uint64)
mask = ~np.isin(sn_oid, oids)
sn_oid = sn_oid[mask][:10]
sn_priors = np.array(train_data[train_data['is_sn'] == 1][names], dtype=np.float32)[mask][:10]

In [4]:
feat_concat = np.vstack([sn_priors, features])
oid_concat = np.hstack([sn_oid, oids])
print(feat_concat.shape)

(557913, 48)


In [5]:
labels = dict(zip(np.arange(0,sn_priors.shape[0]+art_priors.shape[0]),
                  np.hstack([np.ones(art_priors.shape[0]), np.ones(sn_priors.shape[0])*(-1)])))

NameError: name 'art_priors' is not defined

In [8]:
aad = AADForest(random_seed=rs)
aad.fit(features)
np.unique(aad.evaluator.weights)


array([0.01532848])

In [9]:
aad = AADForest(random_seed=rs)
aad.fit_known(features, known_data=sn_priors, known_labels=[Label.ANOMALY for i in range(10)])
np.unique(aad.evaluator.weights)


array([0.01532848])

In [58]:
824211400024886

array([0.01532848])

In [6]:
s = Session(feat_concat, oid_concat,
            known_labels=dict(zip(np.arange(0,sn_priors.shape[0]), np.ones(sn_priors.shape[0])*(-1))),
            on_decision_callbacks= [TerminateAfter(5)],
            decision_callback=viewer_decision_callback,
            model=AADForest(random_seed=rs)) #, n_trees=20, n_spare_trees=80

In [7]:
s.run()

Check https://ztf.snad.space/view/824211400024886 for details
Is 824211400024886 anomaly? [y/N]:

  y


Check https://ztf.snad.space/view/824209200029056 for details
Is 824209200029056 anomaly? [y/N]:

  n


Check https://ztf.snad.space/view/824213200024331 for details
Is 824213200024331 anomaly? [y/N]:

  n


Check https://ztf.snad.space/view/824215300005289 for details
Is 824215300005289 anomaly? [y/N]:

  y


Check https://ztf.snad.space/view/824216200001583 for details
Is 824216200001583 anomaly? [y/N]:

  y


In [14]:
from coniferest.aadforest import AADForest
from coniferest.datasets import single_outlier
from coniferest.onnx.convert import convert, save_onnx_model
from onnxconverter_common.data_types import FloatTensorType

data, _metadata = single_outlier()
data = data.astype(np.float32)
model = AADForest().fit(data)
o = convert(model, initial_types = [('X', FloatTensorType([None, data.shape[1]]))])

In [17]:
_ = save_onnx_model(o, filename='onnx_model/test.onnx')

In [19]:
np.unique(model.evaluator.weights)

array([0.01169771])